In [1]:
# ============================================================
# NRL ANALYTICS WEEKLY — CELL 1: SETUP
# ============================================================

!pip install requests beautifulsoup4 pandas scikit-learn xgboost -q

import requests
import pandas as pd
import numpy as np
import pickle
import json
import os
import re
import warnings
from bs4 import BeautifulSoup
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')

ODDS_API_KEY = "b452724dbd9434d9874cc32fc74f2d16"  # ← your actual key
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

all_teams = ['Broncos','Bulldogs','Cowboys','Dolphins','Dragons','Eels',
             'Knights','Panthers','Rabbitohs','Raiders','Roosters',
             'Sea Eagles','Sharks','Storm','Wests Tigers','Titans','Warriors']

NAME_MAP = {
    'New Zealand Warriors':       'Warriors',
    'North Queensland Cowboys':   'Cowboys',
    'South Sydney Rabbitohs':     'Rabbitohs',
    'Manly Warringah Sea Eagles': 'Sea Eagles',
    'Penrith Panthers':           'Panthers',
    'Melbourne Storm':            'Storm',
    'Brisbane Broncos':           'Broncos',
    'Canterbury Bulldogs':        'Bulldogs',
    'Sydney Roosters':            'Roosters',
    'Cronulla Sutherland Sharks': 'Sharks',
    'Canberra Raiders':           'Raiders',
    'Gold Coast Titans':          'Titans',
    'Newcastle Knights':          'Knights',
    'Parramatta Eels':            'Eels',
    'St George Illawarra Dragons':'Dragons',
    'Wests Tigers':               'Wests Tigers',
    'Dolphins':                   'Dolphins',
}

TEAM_MAP_SC = {
    'Broncos':'Broncos','Raiders':'Raiders','Bulldogs':'Bulldogs',
    'Sharks':'Sharks','Dolphins':'Dolphins','Titans':'Titans',
    'Sea Eagles':'Sea Eagles','Storm':'Storm','Warriors':'Warriors',
    'Knights':'Knights','Panthers':'Panthers','Rabbitohs':'Rabbitohs',
    'Roosters':'Roosters','Cowboys':'Cowboys','Eels':'Eels',
    'Dragons':'Dragons','Tigers':'Wests Tigers',
}

severity_order = {
    "season":0,"indefinite":1,"tbc":2,"6-8 weeks":3,"6 weeks":3,
    "8 weeks":3,"4-6 weeks":4,"4 weeks":4,"5 weeks":4,"2-4 weeks":5,
    "3 weeks":5,"2 weeks":6,"1-2 weeks":7,"1 week":8,"round":9,
}

def severity_score(return_str):
    r = str(return_str).lower().strip()
    for key, score in severity_order.items():
        if key in r: return score
    return 5

def get_winner(row):
    if row['HomeTeamScore'] > row['AwayTeamScore']: return row['HomeTeam']
    elif row['AwayTeamScore'] > row['HomeTeamScore']: return row['AwayTeam']
    return 'Draw'

def get_bye_team(round_df):
    playing = set(round_df['HomeTeam'].tolist() + round_df['AwayTeam'].tolist())
    return [t for t in all_teams if t not in playing]

# ── Model functions ────────────────────────────────────────────
def exp_weighted(values, base=1.5):
    total, weight = 0, 0
    for i, v in enumerate(reversed(values)):
        w = base ** i
        total += v * w
        weight += w
    return total / weight if weight > 0 else 0

def get_diff(g, team):
    return g['HomeTeamScore'] - g['AwayTeamScore'] if g['HomeTeam'] == team \
           else g['AwayTeamScore'] - g['HomeTeamScore']

def get_scored(g, team):
    return g['HomeTeamScore'] if g['HomeTeam'] == team else g['AwayTeamScore']

def get_conceded(g, team):
    return g['AwayTeamScore'] if g['HomeTeam'] == team else g['HomeTeamScore']

def get_result(g, team):
    return 1 if get_diff(g, team) > 0 else 0

def build_team_features(data, team):
    mask   = (data['HomeTeam'] == team) | (data['AwayTeam'] == team)
    recent = data[mask].tail(8)
    if len(recent) < 3: return None
    diffs    = [get_diff(g, team)     for _, g in recent.iterrows()]
    conceded = [get_conceded(g, team) for _, g in recent.iterrows()]
    scored   = [get_scored(g, team)   for _, g in recent.iterrows()]
    results  = [get_result(g, team)   for _, g in recent.iterrows()]
    margins  = [d for d in diffs if d > 0]
    away_g   = data[data['AwayTeam'] == team]
    away_wr  = (away_g['AwayTeamScore'] > away_g['HomeTeamScore']).mean() \
               if len(away_g) >= 3 else 0.42
    atk      = np.mean(scored[-4:]) - np.mean(scored[:-4]) if len(scored) >= 4 else 0
    return {
        'diff_trend':   exp_weighted(diffs),
        'defence':      exp_weighted(conceded),
        'win_rate':     exp_weighted(results),
        'avg_margin':   np.mean(margins) if margins else 0,
        'away_wr':      away_wr,
        'attack_trend': atk,
    }

def get_h2h(data, home, away, n=6):
    mask = (
        ((data['HomeTeam'] == home) & (data['AwayTeam'] == away)) |
        ((data['HomeTeam'] == away) & (data['AwayTeam'] == home))
    )
    matches = data[mask].tail(n)
    if len(matches) == 0: return 0.5
    total_w, home_w = 0, 0
    for i, (_, g) in enumerate(reversed(list(matches.iterrows()))):
        w = 1.5 ** i
        total_w += w
        won = (g['HomeTeam'] == home and g['HomeTeamScore'] > g['AwayTeamScore']) or \
              (g['AwayTeam'] == home and g['AwayTeamScore'] > g['HomeTeamScore'])
        if won: home_w += w
    return home_w / total_w

def predict_match(home, away, all_data, model, scaler, feature_cols):
    hf = build_team_features(all_data, home)
    af = build_team_features(all_data, away)
    if hf is None or af is None: return None
    h2h   = get_h2h(all_data, home, away)
    feats = {
        'diff_trend':        hf['diff_trend']   - af['diff_trend'],
        'defence_diff':      af['defence']       - hf['defence'],
        'win_rate_diff':     hf['win_rate']      - af['win_rate'],
        'margin_diff':       hf['avg_margin']    - af['avg_margin'],
        'away_wr_diff':      hf['away_wr']       - af['away_wr'],
        'attack_trend_diff': hf['attack_trend']  - af['attack_trend'],
        'h2h_rate':          h2h,
        'venue_wr_diff':     0,
        'form_x_margin':     (hf['win_rate'] - af['win_rate']) *
                             (hf['diff_trend'] - af['diff_trend']),
    }
    feat_df     = pd.DataFrame([feats])[feature_cols]
    feat_scaled = scaler.transform(feat_df)
    home_prob   = model.predict_proba(feat_scaled)[0][1]
    away_prob   = 1 - home_prob
    return {
        'home_prob':       round(home_prob * 100, 1),
        'away_prob':       round(away_prob * 100, 1),
        'our_home_odds':   round(1 / home_prob, 2) if home_prob > 0.05 else 99.0,
        'our_away_odds':   round(1 / away_prob, 2) if away_prob > 0.05 else 99.0,
        'predicted_winner': home if home_prob > 0.5 else away,
        'confidence':      round(max(home_prob, away_prob) * 100, 1),
        'is_tossup':       abs(home_prob - 0.5) < 0.10,
    }

# ── HTML helpers ───────────────────────────────────────────────
def form_badges(form_str):
    badges = []
    for ch in str(form_str):
        if ch == 'W':
            badges.append('<span style="display:inline-block;background:#16a34a;color:#fff;padding:3px 7px;border-radius:999px;font-size:12px;font-weight:bold;margin-right:4px;">W</span>')
        elif ch == 'L':
            badges.append('<span style="display:inline-block;background:#dc2626;color:#fff;padding:3px 7px;border-radius:999px;font-size:12px;font-weight:bold;margin-right:4px;">L</span>')
        elif ch == 'D':
            badges.append('<span style="display:inline-block;background:#f59e0b;color:#fff;padding:3px 7px;border-radius:999px;font-size:12px;font-weight:bold;margin-right:4px;">D</span>')
    return "".join(badges)

def severity_icon(return_str):
    score = severity_score(str(return_str))
    if score <= 2:   return '<span style="font-weight:bold;color:#ef4444;">🔴 Long-term</span>'
    elif score <= 4: return '<span style="font-weight:bold;color:#f97316;">🟠 Medium</span>'
    elif score <= 6: return '<span style="font-weight:bold;color:#eab308;">🟡 Short</span>'
    else:            return '<span style="font-weight:bold;color:#22c55e;">🟢 Near return</span>'

def safe_odds(val):
    try:    return f"{float(val):.2f}"
    except: return "N/A"

def matchup_emoji(rating):
    if "Dream"   in rating: return "🔥"
    elif "Good"  in rating: return "✅"
    elif "Neutral" in rating: return "➡️"
    elif "Tough" in rating: return "⚠️"
    elif "Avoid" in rating: return "🚫"
    elif "BYE"   in rating: return "😴"
    return ""

print("✅ Cell 1 ready")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\ryank\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


✅ Cell 1 ready


In [2]:
# ============================================================
# NRL ANALYTICS WEEKLY — CELL 2: GENERATE
# ============================================================
ODDS_API_KEY = "b452724dbd9434d9874cc32fc74f2d16"  # ← paste your key

# ── 1. Match data ──────────────────────────────────────────────
df = pd.DataFrame(requests.get(
    "https://fixturedownload.com/feed/json/nrl-2026", headers=headers).json())
df['Season'] = 2026
completed    = df[df['HomeTeamScore'].notna()].copy()
latest_round = completed['RoundNumber'].max()
next_round   = int(latest_round) + 1
print(f"✅ Round {int(latest_round)} completed | Generating Round {next_round} preview")

# ── 2. Last round results ──────────────────────────────────────
this_round           = completed[completed['RoundNumber'] == latest_round].copy()
this_round['Winner'] = this_round.apply(get_winner, axis=1)
this_round['Score']  = this_round['HomeTeamScore'].astype(int).astype(str) + \
                       ' - ' + this_round['AwayTeamScore'].astype(int).astype(str)
bye_teams            = get_bye_team(this_round)  # last round byes

# ── 3. Ladder ──────────────────────────────────────────────────
all_rounds = completed['RoundNumber'].unique()
bye_counts = {team: 0 for team in all_teams}
for rnd in all_rounds:
    for team in get_bye_team(completed[completed['RoundNumber'] == rnd]):
        bye_counts[team] += 1

# Build all_games WITH RoundNumber so recent games are truly recent
home_g = completed[['RoundNumber','HomeTeam','AwayTeam','HomeTeamScore','AwayTeamScore']].copy()
home_g.columns = ['RoundNumber','Team','Opponent','For','Against']

away_g = completed[['RoundNumber','AwayTeam','HomeTeam','AwayTeamScore','HomeTeamScore']].copy()
away_g.columns = ['RoundNumber','Team','Opponent','For','Against']

all_games = pd.concat([home_g, away_g], ignore_index=True)

# IMPORTANT: sort before using tail()
all_games = all_games.sort_values(['Team', 'RoundNumber']).reset_index(drop=True)

all_games['Win']  = (all_games['For'] > all_games['Against']).astype(int)
all_games['Loss'] = (all_games['For'] < all_games['Against']).astype(int)
all_games['Draw'] = (all_games['For'] == all_games['Against']).astype(int)
all_games['Pts']  = all_games['Win'] * 2 + all_games['Draw']
all_games['Diff'] = all_games['For'] - all_games['Against']

home_games = completed.copy()
home_games['Team'] = home_games['HomeTeam']
home_games['Win']  = (home_games['HomeTeamScore'] > home_games['AwayTeamScore']).astype(int)
away_games = completed.copy()
away_games['Team'] = away_games['AwayTeam']
away_games['Win']  = (away_games['AwayTeamScore'] > away_games['HomeTeamScore']).astype(int)
home_record = home_games.groupby('Team')['Win'].agg(['sum','count']).rename(columns={'sum':'HW','count':'HP'})
away_record = away_games.groupby('Team')['Win'].agg(['sum','count']).rename(columns={'sum':'AW','count':'AP'})

def get_form(team, n=5):
    games = all_games[all_games['Team'] == team].sort_values('RoundNumber').tail(n)
    return ' '.join([
        'W' if r == 1 else ('D' if d == 1 else 'L')
        for r, d in zip(games['Win'], games['Draw'])
    ])

ladder = all_games.groupby('Team').agg(
    Played=('Win','count'), Wins=('Win','sum'), Losses=('Loss','sum'),
    Draws=('Draw','sum'), For=('For','sum'), Against=('Against','sum'),
    Diff=('Diff','sum'), Points=('Pts','sum')
).reset_index()
ladder['Byes'] = ladder['Team'].map(bye_counts)
ladder['Home'] = ladder['Team'].map(lambda t: f"{home_record.loc[t,'HW']}/{home_record.loc[t,'HP']}" if t in home_record.index else '0/0')
ladder['Away'] = ladder['Team'].map(lambda t: f"{away_record.loc[t,'AW']}/{away_record.loc[t,'AP']}" if t in away_record.index else '0/0')
ladder['Form'] = ladder['Team'].map(get_form)
ladder = ladder.sort_values(['Points','Diff'], ascending=False).reset_index(drop=True)
ladder.index += 1
print(f"✅ Ladder built")

# ── 4. Team trajectory ─────────────────────────────────────────
trajectory_data = {}

for team in all_teams:
    team_games = all_games[all_games['Team'] == team].copy()
    team_games = team_games.sort_values('RoundNumber')

    if len(team_games) < 4:
        continue

    recent4 = team_games.tail(4)

    recent_scored = recent4['For'].mean()
    recent_conceded = recent4['Against'].mean()

    season_scored = team_games['For'].mean()
    season_conceded = team_games['Against'].mean()

    attack_trend = recent_scored - season_scored
    defence_trend = season_conceded - recent_conceded
    overall = attack_trend + defence_trend

    if overall > 8:
        arrow = "🔥 Hot"
    elif overall > 4:
        arrow = "📈 Rising"
    elif overall > -4:
        arrow = "➡️ Stable"
    elif overall > -8:
        arrow = "📉 Fading"
    else:
        arrow = "❄️ Cold"

    trajectory_data[team] = {
        'attack_trend': round(float(attack_trend), 1),
        'defence_trend': round(float(defence_trend), 1),
        'recent_scored': round(float(recent_scored), 1),
        'recent_conceded': round(float(recent_conceded), 1),
        'season_scored': round(float(season_scored), 1),
        'season_conceded': round(float(season_conceded), 1),
        'overall': round(float(overall), 1),
        'arrow': arrow,
    }

print("✅ Team trajectories calculated")
# ── 5. Odds ────────────────────────────────────────────────────
odds_response = requests.get(
    "https://api.the-odds-api.com/v4/sports/rugbyleague_nrl/odds",
    params={"apiKey": ODDS_API_KEY, "regions": "au", "markets": "h2h",
            "oddsFormat": "decimal", "bookmakers": "sportsbet"}
)
odds_data = odds_response.json()

if not isinstance(odds_data, list):
    print(f"⚠️ Odds API error: {odds_data}")
    odds_df = pd.DataFrame(columns=['Date','Home','Away','Home Odds','Away Odds','Favourite'])
else:
    odds_rows = []
    for match in odds_data:
        home, away = match['home_team'], match['away_team']
        match_date = match['commence_time'][:10]
        home_odds = away_odds = None
        for bm in match['bookmakers']:
            if bm['key'] == 'sportsbet':
                for o in bm['markets'][0]['outcomes']:
                    if o['name'] == home:   home_odds = o['price']
                    elif o['name'] == away: away_odds = o['price']
        favourite = home if (home_odds and away_odds and home_odds < away_odds) else away
        odds_rows.append({'Date': match_date, 'Home': home, 'Away': away,
                          'Home Odds': home_odds, 'Away Odds': away_odds,
                          'Favourite': favourite})
    odds_df = pd.DataFrame(odds_rows)
    odds_df['DateParsed'] = pd.to_datetime(odds_df['Date'])
    min_date = odds_df['DateParsed'].min()
    max_date = min_date + pd.Timedelta(days=7)
    odds_df  = odds_df[odds_df['DateParsed'] <= max_date].drop_duplicates(
        subset=['Home','Away']).reset_index(drop=True)

print(f"✅ Odds loaded — {len(odds_df)} matches | Remaining: {odds_response.headers.get('x-requests-remaining')}")

# ── 6. Model predictions ───────────────────────────────────────
all_hist = []
for year in [2021,2022,2023,2024,2025,2026]:
    r = requests.get(f"https://fixturedownload.com/feed/json/nrl-{year}", headers=headers)
    if r.status_code == 200:
        dy = pd.DataFrame(r.json())
        dy['Season'] = year
        dy = dy[dy['HomeTeamScore'].notna()].copy()
        all_hist.append(dy)
all_data = pd.concat(all_hist, ignore_index=True)
all_data = all_data.sort_values(['Season','RoundNumber','MatchNumber']).reset_index(drop=True)

feature_cols = ['diff_trend','defence_diff','win_rate_diff','margin_diff',
                'away_wr_diff','attack_trend_diff','h2h_rate','venue_wr_diff','form_x_margin']

if os.path.exists("nrl_model.pkl"):
    with open("nrl_model.pkl","rb") as f:
        saved = pickle.load(f)
    nrl_model    = saved['model']
    nrl_scaler   = saved['scaler']
    nrl_features = saved['feature_cols']
    print(f"✅ Model loaded: {saved['model_name']}")
else:
    print("⚠️ No model found — retrain first")
    nrl_model = nrl_scaler = nrl_features = None

predictions = []
if nrl_model and len(odds_df) > 0:
    for _, fixture in odds_df.iterrows():
        home_std = NAME_MAP.get(fixture['Home'], fixture['Home'])
        away_std = NAME_MAP.get(fixture['Away'], fixture['Away'])
        pred = predict_match(home_std, away_std, all_data, nrl_model, nrl_scaler, nrl_features)
        if pred is None: continue
        sb_home    = fixture['Home Odds']
        sb_away    = fixture['Away Odds']
        home_edge  = round(float(sb_home) - pred['our_home_odds'], 2) if sb_home and str(sb_home) != 'nan' else 0
        away_edge  = round(float(sb_away) - pred['our_away_odds'], 2) if sb_away and str(sb_away) != 'nan' else 0
        value_side = value_odds = value_edge = None
        if home_edge >= 0.20:
            value_side, value_odds, value_edge = home_std, sb_home, home_edge
        if away_edge >= 0.20 and away_edge > home_edge:
            value_side, value_odds, value_edge = away_std, sb_away, away_edge
        predictions.append({
            'home': home_std, 'away': away_std,
            'home_prob': pred['home_prob'], 'away_prob': pred['away_prob'],
            'our_home_odds': pred['our_home_odds'], 'our_away_odds': pred['our_away_odds'],
            'sb_home': sb_home, 'sb_away': sb_away,
            'predicted_winner': pred['predicted_winner'],
            'confidence': pred['confidence'], 'is_tossup': pred['is_tossup'],
            'value_side': value_side, 'value_odds': value_odds, 'value_edge': value_edge,
        })
pred_df = pd.DataFrame(predictions)
print(f"✅ Predictions generated for {len(pred_df)} matches")

# ── 7. Injuries ────────────────────────────────────────────────
inj_r  = requests.get("https://www.zerotackle.com/nrl/injuries-suspensions/", headers=headers)
soup   = BeautifulSoup(inj_r.text, "html.parser")
tables = soup.find_all("table")
injuries = []
for table in tables:
    team_name = "Unknown"
    for sibling in table.previous_siblings:
        if hasattr(sibling, 'get_text'):
            text = sibling.get_text(strip=True)
            if text: team_name = text; break
    for row in table.find_all("tr", class_="table-row-border"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            injuries.append({
                "Team":   team_name,
                "Player": cells[1].get_text(separator=" ", strip=True),
                "Injury": cells[2].get_text(strip=True),
                "Return": cells[3].get_text(strip=True) if len(cells) > 3 else "TBC"
            })
injuries_df = pd.DataFrame(injuries)
print(f"✅ Injuries: {len(injuries_df)} players")

# ── 8. SuperCoach fantasy tips ─────────────────────────────────
sc_resp    = requests.get(
    f"https://supercoach.com.au/2026/api/nrl/draft/v1/players-cf?embed=notes,odds,stats&round={int(latest_round)}",
    headers=headers)
sc_players = sc_resp.json()

sc_rows = []
for p in sc_players:
    if not p.get('active'): continue
    team  = p.get('team', {})
    notes = p.get('notes', [])
    sc_rows.append({
        'name':          f"{p['first_name']} {p['last_name']}",
        'team':          team.get('name',''),
        'team_abbrev':   team.get('abbrev',''),
        'games':         p.get('previous_games', 0) or 0,
        'avg_2025':      p.get('previous_average', 0) or 0,
        'draft_rank':    p.get('predraft_rank', 999) or 999,
        'injury_status': p.get('injury_suspension_status', None),
        'injury_text':   p.get('injury_suspension_status_text','') or '',
        'latest_note':   notes[0]['note'] if notes else '',
        'note_date':     notes[0]['created_on'][:10] if notes else '',
    })
sc_df = pd.DataFrame(sc_rows)
sc_df = sc_df[sc_df['avg_2025'] > 0].copy()
sc_df['team_std'] = sc_df['team'].map(TEAM_MAP_SC).fillna(sc_df['team'])

# Defensive strength (percentile based)
home_con = completed.groupby('HomeTeam')['AwayTeamScore'].mean()
away_con = completed.groupby('AwayTeam')['HomeTeamScore'].mean()
team_def = {}
for team in completed['HomeTeam'].unique():
    h = home_con.get(team, 0)
    a = away_con.get(team, 0)
    team_def[team] = round((h + a) / 2, 1)

def_values = list(team_def.values())
p25 = np.percentile(def_values, 25)
p50 = np.percentile(def_values, 50)
p75 = np.percentile(def_values, 75)
p90 = np.percentile(def_values, 90)

# Next round fixtures
next_fix = df[df['RoundNumber'] == next_round].copy()
opp_lookup = {}
for _, fix in next_fix.iterrows():
    opp_lookup[fix['HomeTeam']] = {'opponent': fix['AwayTeam'],  'home_away': 'HOME', 'opp_def': team_def.get(fix['AwayTeam'], 20)}
    opp_lookup[fix['AwayTeam']] = {'opponent': fix['HomeTeam'], 'home_away': 'AWAY', 'opp_def': team_def.get(fix['HomeTeam'], 20)}

sc_df['opponent']  = sc_df['team_std'].map(lambda t: opp_lookup.get(t,{}).get('opponent','BYE'))
sc_df['home_away'] = sc_df['team_std'].map(lambda t: opp_lookup.get(t,{}).get('home_away','-'))
sc_df['opp_def']   = sc_df['team_std'].map(lambda t: opp_lookup.get(t,{}).get('opp_def', 20))
sc_df['has_bye']   = sc_df['opponent'] == 'BYE'

def matchup_rating(opp_def, has_bye):
    if has_bye:          return "😴 BYE"
    if opp_def >= p90:   return "🔥 Dream matchup"
    elif opp_def >= p75: return "✅ Good matchup"
    elif opp_def >= p50: return "➡️ Neutral"
    elif opp_def >= p25: return "⚠️ Tough matchup"
    else:                return "🚫 Avoid"

sc_df['matchup'] = sc_df.apply(lambda r: matchup_rating(r['opp_def'], r['has_bye']), axis=1)

# Cap 2 players per team
def cap_per_team(df, max_per_team=2):
    result, counts = [], {}
    for _, row in df.iterrows():
        t = row['team_std']
        if counts.get(t, 0) < max_per_team:
            result.append(row)
            counts[t] = counts.get(t, 0) + 1
    return pd.DataFrame(result)

top_plays = cap_per_team(sc_df[
    (sc_df['avg_2025'] >= 60) &
    (~sc_df['has_bye']) &
    (sc_df['injury_status'].isna() | (sc_df['injury_status'] == '')) &
    (sc_df['opp_def'] >= p50)
].sort_values(['opp_def','avg_2025'], ascending=[False,False]).head(20))

value_picks = cap_per_team(sc_df[
    (sc_df['avg_2025'] >= 40) &
    (sc_df['avg_2025'] < 65) &
    (~sc_df['has_bye']) &
    (sc_df['injury_status'].isna() | (sc_df['injury_status'] == '')) &
    (sc_df['opp_def'] >= p75)
].sort_values(['opp_def','avg_2025'], ascending=[False,False]).head(20))

avoid_list = sc_df[
    (sc_df['avg_2025'] >= 55) &
    (sc_df['has_bye'] | (sc_df['opp_def'] < p25))
].sort_values('avg_2025', ascending=False).head(6)

noted = sc_df[
    (sc_df['latest_note'] != '') &
    (sc_df['avg_2025'] >= 65) &
    (~sc_df['has_bye'])
].sort_values('avg_2025', ascending=False).head(5)

bye_teams_next = [t for t in all_teams if t not in set(
    next_fix['HomeTeam'].tolist() + next_fix['AwayTeam'].tolist())]

# Key injured players (SC avg >= 60)
key_players = sc_df[sc_df['avg_2025'] >= 60][['name','team_std','avg_2025']].copy()
_inj = injuries_df.copy()
if not _inj.empty:
    _inj['SeverityScore'] = _inj['Return'].apply(severity_score)
    _inj = _inj.sort_values(['Team','SeverityScore'], ascending=[True,True]).reset_index(drop=True)

injured_key = pd.DataFrame()
if not _inj.empty and not key_players.empty:
    injured_key = _inj.merge(
        key_players, left_on='Player', right_on='name', how='inner'
    ).sort_values('avg_2025', ascending=False).drop_duplicates('Player')

print(f"✅ SuperCoach tips ready | Key injured players: {len(injured_key)}")

# ── Team colours for CSS badges (always works, no external deps) ──
TEAM_COLOURS = {
    'Broncos':     ('#800000', '#FFD700', 'BRO'),
    'Raiders':     ('#01B65C', '#000000', 'CAN'),
    'Bulldogs':    ('#0033A0', '#FFFFFF', 'BUL'),
    'Sharks':      ('#00AEEF', '#000000', 'CRO'),
    'Dolphins':    ('#CC0000', '#003087', 'DOL'),
    'Titans':      ('#009FDF', '#000000', 'GCT'),
    'Sea Eagles':  ('#6F263D', '#FFFFFF', 'MNL'),
    'Storm':       ('#4F2D7F', '#FFD700', 'MEL'),
    'Warriors':    ('#000000', '#808080', 'NZL'),
    'Knights':     ('#003B8E', '#C0C0C0', 'NEW'),
    'Panthers':    ('#000000', '#FFFFFF', 'PEN'),
    'Rabbitohs':   ('#006A4E', '#FF0000', 'SOU'),
    'Roosters':    ('#003087', '#FFD700', 'SYD'),
    'Cowboys':     ('#003087', '#FFD700', 'NQL'),
    'Eels':        ('#0033CC', '#FFD700', 'PAR'),
    'Dragons':     ('#CC0000', '#FFFFFF', 'SGI'),
    'Wests Tigers':('#FF6600', '#000000', 'WST'),
}

# ── Fetch real logo URLs from NRL draw API ────────────────────
nrl_logos = {}
try:
    for rnd_num in range(max(1, int(latest_round)-2), int(latest_round)+1):
        draw_r = requests.get(
            f"https://www.nrl.com/draw/data?competition=111&season=2026&round={rnd_num}",
            headers=headers, timeout=10
        )
        if draw_r.status_code == 200:
            draw_data = draw_r.json()
            for fixture in draw_data.get('fixtures', []):
                for side in ['homeTeam', 'awayTeam']:
                    team_data = fixture.get(side, {})
                    nick = team_data.get('nickName', '')
                    theme = team_data.get('theme', {})
                    logos = theme.get('logos', {})
                    badge_url = logos.get('badge.svg', logos.get('badge.png', ''))
                    if nick and badge_url:
                        # Build full URL
                        full_url = f"https://www.nrl.com/siteassets/2026/teams/{theme.get('key','')}/{badge_url}" \
                                   if not badge_url.startswith('http') else badge_url
                        nrl_logos[nick] = full_url
    print(f"✅ Fetched {len(nrl_logos)} team logos from NRL API")
except Exception as e:
    print(f"⚠️ Could not fetch NRL logos: {e}")

def team_badge(team_name, size=44):
    """
    Try NRL API logo first, fall back to CSS colour badge.
    CSS badge always works — no external dependency.
    """
    colours = TEAM_COLOURS.get(team_name, ('#374151', '#FFFFFF', team_name[:3].upper()))
    bg, fg, abbr = colours

    # CSS fallback badge — always visible
    css_badge = f'''<div style="
        width:{size}px;height:{size}px;border-radius:8px;
        background:{bg};color:{fg};
        display:flex;align-items:center;justify-content:center;
        font-size:{int(size*0.28)}px;font-weight:900;
        flex-shrink:0;letter-spacing:-0.5px;
        border:2px solid rgba(0,0,0,0.1);">
        {abbr}
    </div>'''

    # If we got a real logo URL, wrap in an img with CSS fallback
    logo_url = nrl_logos.get(team_name, '')
    if logo_url:
        return f'''<div style="width:{size}px;height:{size}px;flex-shrink:0;position:relative;">
            <img src="{logo_url}"
                 alt="{team_name}"
                 style="width:{size}px;height:{size}px;object-fit:contain;"
                 onerror="this.style.display='none';this.nextElementSibling.style.display='flex';">
            <div style="display:none;width:{size}px;height:{size}px;border-radius:8px;
                background:{bg};color:{fg};
                align-items:center;justify-content:center;
                font-size:{int(size*0.28)}px;font-weight:900;
                border:2px solid rgba(0,0,0,0.1);">
                {abbr}
            </div>
        </div>'''

    return css_badge

# ── Helper functions ───────────────────────────────────────────
def safe_odds_display(val):
    try:
        f = float(val)
        if f != f: return "TBA"
        return f"${f:.2f}"
    except:
        return "TBA"

def clean_team_name(name):
    return NAME_MAP.get(str(name).strip(), str(name).strip())

# ── 9. Build HTML ──────────────────────────────────────────────
print("Building newsletter HTML...")

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>NRL Analytics Weekly — Round {next_round}</title>
<style>
  body {{ font-family: Arial, Helvetica, sans-serif; background:#f1f5f9; margin:0; padding:20px; color:#111827; }}
  .wrap {{ max-width:960px; margin:0 auto; }}
  .card {{ background:#fff; border:1px solid #e5e7eb; border-radius:16px; padding:24px; margin-bottom:24px; }}
  .header {{ background:linear-gradient(135deg,#0f172a,#1e3a5f); color:#fff; border-radius:16px; padding:32px 24px; text-align:center; margin-bottom:24px; }}
  .header h1 {{ margin:8px 0; font-size:2.2rem; }}
  h2 {{ font-size:1.4rem; margin:0 0 16px 0; color:#0f172a; border-bottom:2px solid #e5e7eb; padding-bottom:8px; }}
  h3 {{ font-size:1.05rem; margin:16px 0 10px; }}
  table {{ width:100%; border-collapse:collapse; font-size:14px; }}
  th {{ background:#f8fafc; padding:10px 12px; border-bottom:2px solid #e5e7eb; text-align:left; font-weight:600; color:#374151; }}
  td {{ padding:10px 12px; border-bottom:1px solid #f1f5f9; }}
  tr:hover td {{ background:#f8fafc; }}
  .badge {{ display:inline-block; padding:3px 10px; border-radius:999px; font-size:12px; font-weight:bold; }}
  .badge-green  {{ background:#dcfce7; color:#166534; }}
  .badge-red    {{ background:#fee2e2; color:#991b1b; }}
  .badge-yellow {{ background:#fef3c7; color:#92400e; }}
  .badge-blue   {{ background:#dbeafe; color:#1e40af; }}
  .pill-w {{ display:inline-block;background:#16a34a;color:#fff;padding:3px 7px;border-radius:999px;font-size:11px;font-weight:bold;margin-right:2px; }}
  .pill-l {{ display:inline-block;background:#dc2626;color:#fff;padding:3px 7px;border-radius:999px;font-size:11px;font-weight:bold;margin-right:2px; }}
  .pill-d {{ display:inline-block;background:#f59e0b;color:#fff;padding:3px 7px;border-radius:999px;font-size:11px;font-weight:bold;margin-right:2px; }}
  .match-card {{ display:grid;grid-template-columns:1fr auto 1fr;align-items:center;padding:14px 8px;border-bottom:1px solid #f1f5f9;gap:8px; }}
  .match-score {{ text-align:center;font-size:1.5rem;font-weight:800;color:#0f172a;white-space:nowrap; }}
  .match-team {{ font-size:14px; }}
  .match-team.winner {{ font-weight:800; }}
  .match-team.home {{ text-align:right; }}
  .match-team.away {{ text-align:left; }}
  .traj-up {{ color:#16a34a;font-weight:bold; }}
  .traj-dn {{ color:#dc2626;font-weight:bold; }}
  .traj-nt {{ color:#6b7280; }}
  .section-intro {{ color:#6b7280;font-size:13px;margin:-8px 0 16px 0;line-height:1.6; }}
  .sc-box {{ background:#f0fdf4;border:1px solid #86efac;border-radius:10px;padding:12px 16px;margin-bottom:8px; }}
  .sc-box.value {{ background:#eff6ff;border-color:#93c5fd; }}
  .sc-avoid {{ background:#fef2f2;border:1px solid #fca5a5;border-radius:10px;padding:12px 16px;margin-bottom:8px; }}
  .note-box {{ background:#f8fafc;border-left:3px solid #64b5f6;padding:10px 14px;margin-bottom:10px;border-radius:4px;font-size:13px; }}
  .key-injury {{ background:#fff7ed;border:1px solid #fed7aa;border-radius:10px;padding:12px 16px;margin-bottom:8px;display:flex;justify-content:space-between;align-items:center; }}
  .bye-banner {{ background:#eff6ff;border:1px solid #bfdbfe;padding:12px 18px;border-radius:12px;margin-bottom:24px;font-size:15px; }}
  .footer {{ text-align:center;color:#9ca3af;font-size:12px;padding:20px 0; }}
</style>
</head>
<body>
<div class="wrap">

<!-- HEADER -->
<div class="header">
  <div style="opacity:0.8;font-size:0.9rem;letter-spacing:1px;text-transform:uppercase;">Weekly Data-Driven Rugby League Analysis</div>
  <h1>🏉 NRL Analytics Weekly</h1>
  <div style="font-size:1.2rem;font-weight:600;margin-top:4px;">Round {next_round} — 2026 Season</div>
  <div style="margin-top:8px;opacity:0.6;font-size:12px;">Model accuracy: 62.4% on 2025 season (213 matches)</div>
</div>



<!-- ROUND RESULTS -->
<div class="card">
  <h2>📋 Round {int(latest_round)} Results</h2>
  {"" if not bye_teams else f'<p class="section-intro">😴 Teams on bye Round {int(latest_round)}: {", ".join(bye_teams)}</p>'}
"""

for _, row in this_round.iterrows():
    home_win = row['Winner'] == row['HomeTeam']
    away_win = row['Winner'] == row['AwayTeam']

    # Darker green highlight for winner side
    home_bg = "background:linear-gradient(90deg,#bbf7d0,#f0fdf4);" if home_win else "background:#fafafa;"
    away_bg = "background:linear-gradient(270deg,#bbf7d0,#f0fdf4);" if away_win else "background:#fafafa;"

    home_name_style = "font-weight:800;color:#0f172a;font-size:15px;" if home_win else "font-weight:500;color:#6b7280;font-size:15px;"
    away_name_style = "font-weight:800;color:#0f172a;font-size:15px;" if away_win else "font-weight:500;color:#6b7280;font-size:15px;"

    home_badge = team_badge(row['HomeTeam'], 44)
    away_badge = team_badge(row['AwayTeam'], 44)

    html += f"""
  <div style="display:grid;grid-template-columns:1fr 130px 1fr;
              align-items:stretch;border-bottom:1px solid #e5e7eb;overflow:hidden;">

    <!-- Home team -->
    <div style="display:flex;align-items:center;justify-content:flex-end;
                gap:12px;padding:16px 12px;{home_bg}">
      <div style="text-align:right;">
        <div style="{home_name_style}">{row['HomeTeam']}</div>
        {"<div style='font-size:10px;font-weight:800;color:#15803d;margin-top:3px;letter-spacing:0.5px;'>✓ WINNER</div>" if home_win else ""}
      </div>
      {home_badge}
    </div>

    <!-- Score centre -->
    <div style="display:flex;flex-direction:column;align-items:center;
                justify-content:center;padding:12px 8px;
                background:#0f172a;color:#fff;">
      <div style="font-size:1.6rem;font-weight:900;letter-spacing:-1px;line-height:1;">
        {row['Score']}
      </div>
      <div style="font-size:9px;opacity:0.6;margin-top:4px;
                  text-transform:uppercase;letter-spacing:1.5px;">
        Full Time
      </div>
    </div>

    <!-- Away team -->
    <div style="display:flex;align-items:center;justify-content:flex-start;
                gap:12px;padding:16px 12px;{away_bg}">
      {away_badge}
      <div style="text-align:left;">
        <div style="{away_name_style}">{row['AwayTeam']}</div>
        {"<div style='font-size:10px;font-weight:800;color:#15803d;margin-top:3px;letter-spacing:0.5px;'>✓ WINNER</div>" if away_win else ""}
      </div>
    </div>

  </div>"""

html += f"""
</div>

<!-- NEXT ROUND BYE BANNER -->
{"" if not bye_teams_next else f'<div class="bye-banner"><strong>😴 Bye Round {next_round}:</strong> {", ".join(bye_teams_next)}</div>'}

<!-- LADDER -->
<div class="card">
  <h2>🏆 Ladder — Top 8</h2>
"""
html += f'  <p class="section-intro">After Round {int(latest_round)}. Green rows = top 8 finals position.</p>'
html += """
  <table>
    <thead><tr>
      <th>Pos</th><th>Team</th><th>P</th><th>W</th><th>L</th>
      <th>Pts</th><th>Diff</th><th>Home</th><th>Away</th><th>Form</th>
    </tr></thead><tbody>
"""

for pos, (_, row) in enumerate(ladder.head(8).iterrows(), 1):
    form_html = ""
    for ch in str(row['Form']):
        if ch == 'W': form_html += '<span class="pill-w">W</span>'
        elif ch == 'L': form_html += '<span class="pill-l">L</span>'
        elif ch == 'D': form_html += '<span class="pill-d">D</span>'

    bg = "background:#f0fdf4;" if pos <= 8 else ""
    badge = team_badge(row['Team'], 30)

    html += f"""
    <tr style="{bg}">
      <td style="font-weight:bold;text-align:center;">{pos}</td>

      <td style="font-weight:600;">
        <div style="display:flex;align-items:center;gap:8px;">
          {badge}
          <span>{row['Team']}</span>
        </div>
      </td>

      <td style="text-align:center;">{int(row['Played'])}</td>
      <td style="text-align:center;">{int(row['Wins'])}</td>
      <td style="text-align:center;">{int(row['Losses'])}</td>
      <td style="text-align:center;font-weight:bold;">{int(row['Points'])}</td>
      <td style="text-align:center;">{int(row['Diff'])}</td>
      <td style="text-align:center;font-size:12px;">{row['Home']}</td>
      <td style="text-align:center;font-size:12px;">{row['Away']}</td>
      <td>{form_html}</td>
    </tr>"""

html += """
  </tbody></table>
</div>

<!-- TEAM TRAJECTORY -->
<div class="card">
  <h2>📈 Team Trajectory — Last 4 Rounds</h2>
  <p class="section-intro">
    Compares each team's last 4 games vs season average. Positive defence = conceding less recently.
  </p>

  <table>
    <thead><tr>
      <th>Team</th>
      <th style="text-align:center;">Status</th>
      <th style="text-align:center;">Overall</th>
      <th style="text-align:center;">Attack</th>
      <th style="text-align:center;">Defence</th>
      <th style="text-align:center;">Recent For</th>
      <th style="text-align:center;">Recent Against</th>
    </tr></thead><tbody>
"""

traj_sorted = sorted(trajectory_data.items(), key=lambda x: x[1]['overall'], reverse=True)

for team, t in traj_sorted:
    badge = team_badge(team, 30)

    atk_col = "traj-up" if t['attack_trend'] > 3 else "traj-dn" if t['attack_trend'] < -3 else "traj-nt"
    def_col = "traj-up" if t['defence_trend'] > 3 else "traj-dn" if t['defence_trend'] < -3 else "traj-nt"

    overall_bg = (
        "#dcfce7" if t['overall'] > 4 else
        "#fee2e2" if t['overall'] < -4 else
        "#f3f4f6"
    )

    overall_col = (
        "#166534" if t['overall'] > 4 else
        "#991b1b" if t['overall'] < -4 else
        "#374151"
    )

    atk_sign = "+" if t['attack_trend'] >= 0 else ""
    def_sign = "+" if t['defence_trend'] >= 0 else ""
    overall_sign = "+" if t['overall'] >= 0 else ""

    html += f"""
    <tr>
      <td style="font-weight:600;">
        <div style="display:flex;align-items:center;gap:8px;">
          {badge}
          <span>{team}</span>
        </div>
      </td>

      <td style="text-align:center;font-weight:700;">{t['arrow']}</td>

      <td style="text-align:center;">
        <span style="background:{overall_bg};color:{overall_col};
                     padding:5px 10px;border-radius:999px;
                     font-size:12px;font-weight:800;">
          {overall_sign}{t['overall']}
        </span>
      </td>

      <td style="text-align:center;" class="{atk_col}">
        {atk_sign}{t['attack_trend']}
      </td>

      <td style="text-align:center;" class="{def_col}">
        {def_sign}{t['defence_trend']}
      </td>

      <td style="text-align:center;">{t['recent_scored']}</td>
      <td style="text-align:center;">{t['recent_conceded']}</td>
    </tr>"""

html += """
  </tbody></table>
</div>

<!-- SPORTSBET ODDS -->
<div class="card">
"""
html += f'  <h2>🎲 Round {next_round} — Sportsbet Odds</h2>'
html += """
  <p class="section-intro">
    Current head-to-head odds. ⭐ = market favourite. TBA = not yet published.
  </p>

  <table>
    <thead><tr>
      <th>Date</th>
      <th>Home</th>
      <th style="text-align:center;">Home Odds</th>
      <th style="text-align:center;">Away Odds</th>
      <th>Away</th>
      <th style="text-align:center;">Favourite</th>
    </tr></thead><tbody>
"""

if len(odds_df) == 0:
    html += '<tr><td colspan="6" style="text-align:center;color:#6b7280;padding:20px;">Odds not yet published for this round.</td></tr>'
else:
  for _, row in odds_df.iterrows():
      home = clean_team_name(row['Home'])
      away = clean_team_name(row['Away'])
      fav = clean_team_name(row['Favourite'])

      home_badge = team_badge(home, 30)
      away_badge = team_badge(away, 30)

      fav_home = fav == home
      fav_away = fav == away

      home_style = "font-weight:800;color:#0f172a;" if fav_home else "font-weight:500;color:#64748b;"
      away_style = "font-weight:800;color:#0f172a;" if fav_away else "font-weight:500;color:#64748b;"

      home_odds_bg = "#fef3c7" if fav_home else "#f8fafc"
      away_odds_bg = "#fef3c7" if fav_away else "#f8fafc"

      html += f"""
      <tr>
        <td style="font-size:12px;color:#475569;">{row['Date']}</td>

        <td>
          <div style="display:flex;align-items:center;gap:8px;{home_style}">
            {home_badge}
            <span>{home}</span>
          </div>
        </td>

        <td style="text-align:center;">
          <span style="background:{home_odds_bg};padding:5px 11px;
                      border-radius:999px;font-weight:800;font-size:13px;">
            {safe_odds_display(row['Home Odds'])}
          </span>
        </td>

        <td style="text-align:center;">
          <span style="background:{away_odds_bg};padding:5px 11px;
                      border-radius:999px;font-weight:800;font-size:13px;">
            {safe_odds_display(row['Away Odds'])}
          </span>
        </td>

        <td>
          <div style="display:flex;align-items:center;gap:8px;{away_style}">
            {away_badge}
            <span>{away}</span>
          </div>
        </td>

        <td style="text-align:center;">
          <span class="badge badge-yellow">⭐ {fav}</span>
        </td>
      </tr>"""

html += f"""
  </tbody></table>
</div>



<!-- MODEL PREDICTIONS -->
<div class="card">
  <h2>🤖 Round {next_round} — Our Model vs Sportsbet</h2>

  <p class="section-intro">
    Logistic regression trained on 2021–2024 NRL data. 
    <strong>62.4% accuracy</strong> on 2025 (213 matches).<br>
    👀 Value bet = Sportsbet offering $0.20+ more than our model's fair value.
    🎲 Toss-up = model within 10% of 50/50.
  </p>
"""

if len(pred_df) == 0:
    html += """
    <div style="text-align:center;color:#6b7280;padding:24px;">
      No predictions available — check model and odds data.
    </div>
    """
else:
    html += """
    <div style="display:flex;flex-direction:column;gap:12px;margin-top:14px;">
    """

    for _, p in pred_df.iterrows():
        home = clean_team_name(p["home"])
        away = clean_team_name(p["away"])
        pick = clean_team_name(p["predicted_winner"])

        home_badge = team_badge(home, 32)
        away_badge = team_badge(away, 32)

        conf = float(p["confidence"])

        if conf >= 70:
            conf_bg, conf_col, conf_label, border_col = "#dcfce7", "#166534", "Strong", "#22c55e"
        elif conf >= 60:
            conf_bg, conf_col, conf_label, border_col = "#fef3c7", "#92400e", "Medium", "#f59e0b"
        else:
            conf_bg, conf_col, conf_label, border_col = "#dbeafe", "#1e40af", "Low", "#3b82f6"

        signals = []

        if p["value_side"]:
            value_side = clean_team_name(p["value_side"])
            signals.append(
                f'<span style="background:#fef3c7;color:#92400e;padding:5px 10px;border-radius:999px;font-size:12px;font-weight:800;">👀 VALUE: {value_side} +${p["value_edge"]}</span>'
            )

        if p["is_tossup"]:
            signals.append(
                '<span style="background:#dbeafe;color:#1e40af;padding:5px 10px;border-radius:999px;font-size:12px;font-weight:800;">🎲 Toss-up</span>'
            )

        signal_html = " ".join(signals) if signals else '<span style="color:#9ca3af;font-size:12px;">No clear edge</span>'

        html += f"""
    <div style="
      border:1px solid #e5e7eb;
      border-left:5px solid {border_col};
      border-radius:14px;
      background:#ffffff;
      overflow:hidden;
      box-shadow:0 2px 8px rgba(15,23,42,0.05);
    ">

      <div style="
        display:flex;
        align-items:center;
        justify-content:space-between;
        gap:12px;
        padding:14px 16px;
        background:#f8fafc;
        border-bottom:1px solid #e5e7eb;
      ">

        <div style="display:flex;align-items:center;gap:9px;min-width:0;">
          {home_badge}
          <span style="font-weight:800;color:#0f172a;">{home}</span>
          <span style="color:#94a3b8;font-size:12px;font-weight:800;">vs</span>
          {away_badge}
          <span style="font-weight:800;color:#0f172a;">{away}</span>
        </div>

        <span style="background:{conf_bg};color:{conf_col};
                     padding:6px 11px;border-radius:999px;
                     font-size:12px;font-weight:900;white-space:nowrap;">
          {conf}% · {conf_label}
        </span>
      </div>

      <div style="
        display:grid;
        grid-template-columns:repeat(4, 1fr);
        gap:10px;
        padding:14px 16px;
      ">

        <div>
          <div style="font-size:11px;color:#64748b;font-weight:800;margin-bottom:5px;">OUR PICK</div>
          <div style="font-weight:900;color:#1e3a8a;background:#eef2ff;
                      display:inline-block;padding:6px 12px;border-radius:999px;">
            {pick}
          </div>
        </div>

        <div>
          <div style="font-size:11px;color:#64748b;font-weight:800;margin-bottom:5px;">OUR ODDS</div>
          <div style="font-weight:800;color:#0f172a;">
            ${p["our_home_odds"]} / ${p["our_away_odds"]}
          </div>
        </div>

        <div>
          <div style="font-size:11px;color:#64748b;font-weight:800;margin-bottom:5px;">SPORTSBET</div>
          <div style="font-weight:800;color:#0f172a;">
            {safe_odds_display(p["sb_home"])} / {safe_odds_display(p["sb_away"])}
          </div>
        </div>

        <div>
          <div style="font-size:11px;color:#64748b;font-weight:800;margin-bottom:5px;">SIGNAL</div>
          <div style="display:flex;gap:6px;flex-wrap:wrap;">
            {signal_html}
          </div>
        </div>

      </div>
    </div>
        """

    html += """
    </div>
    """

html += """
  <div style="font-size:11px;color:#9ca3af;margin-top:12px;">
    ⚠️ Not financial advice. Bet responsibly | 18+ | gamblinghelponline.org.au
  </div>
</div>




<!-- KEY INJURED PLAYERS -->
<div class="card">
  <h2>⚡ Key Players Currently Injured</h2>
  <p class="section-intro">High-profile players (60+ SuperCoach avg) currently unavailable. Critical info for Draft owners.</p>
"""

if len(injured_key) == 0:
    html += '<p style="color:#6b7280;font-size:14px;">No high-profile injuries currently listed.</p>'
else:
    for _, row in injured_key.iterrows():
        sev = severity_score(row['Return'])
        if sev <= 2:   sev_col, sev_label = "#ef4444", "🔴 Long-term"
        elif sev <= 4: sev_col, sev_label = "#f97316", "🟠 Medium"
        elif sev <= 6: sev_col, sev_label = "#eab308", "🟡 Short-term"
        else:          sev_col, sev_label = "#22c55e", "🟢 Near return"
        html += f"""
  <div class="key-injury">
    <div>
      <strong style="font-size:15px;">{row['Player']}</strong>
      <span style="color:#6b7280;font-size:13px;margin-left:8px;">({row['Team']})</span>
      <div style="font-size:13px;color:#374151;margin-top:3px;">
        {row['Injury']} — Return: <strong>{row['Return']}</strong>
      </div>
    </div>
    <div style="text-align:right;flex-shrink:0;margin-left:12px;">
      <span class="badge" style="background:#fff7ed;color:{sev_col};">{sev_label}</span>
      <div style="font-size:12px;color:#6b7280;margin-top:4px;">{row['avg_2025']:.0f} SC avg</div>
    </div>
  </div>"""

html += """
</div>



<!-- SUPERCOACH FANTASY TIPS -->
<div class="card">
"""
html += f'  <h2>🏅 SuperCoach Draft Tips — Round {next_round}</h2>'
html += """
  <p class="section-intro">
    Based on 2025 season averages + 2026 opponent defensive strength (percentile-based).<br>
    🔥 Dream · ✅ Good · ➡️ Neutral · ⚠️ Tough · 🚫 Avoid · 😴 BYE<br>
    <strong>Always check Thursday team lists before locking in picks.</strong>
  </p>

  <h3 style="color:#166534;">⭐ Top Plays This Week</h3>
"""

for _, p in top_plays.iterrows():
    emoji = matchup_emoji(p['matchup'])
    html += f"""
  <div class="sc-box">
    <div style="display:flex;justify-content:space-between;align-items:center;">
      <div>
        <strong style="font-size:15px;">{p['name']}</strong>
        <span class="badge badge-blue" style="margin-left:8px;">{p['team_abbrev']}</span>
      </div>
      <span class="badge badge-green">{p['avg_2025']:.0f} SC avg</span>
    </div>
    <div style="font-size:13px;color:#374151;margin-top:6px;">
      {emoji} vs <strong>{p['opponent']}</strong> ({p['home_away']}) — {p['matchup']}
    </div>
  </div>"""

html += """
  <h3 style="color:#1e40af;">💎 Value Picks</h3>
"""

for _, p in value_picks.iterrows():
    emoji = matchup_emoji(p['matchup'])
    html += f"""
  <div class="sc-box value">
    <div style="display:flex;justify-content:space-between;align-items:center;">
      <div>
        <strong style="font-size:15px;">{p['name']}</strong>
        <span class="badge badge-blue" style="margin-left:8px;">{p['team_abbrev']}</span>
      </div>
      <span class="badge badge-blue">{p['avg_2025']:.0f} SC avg</span>
    </div>
    <div style="font-size:13px;color:#374151;margin-top:6px;">
      {emoji} vs <strong>{p['opponent']}</strong> ({p['home_away']}) — {p['matchup']}
    </div>
  </div>"""

html += """
  <h3 style="color:#991b1b;">⚠️ Watch Out — BYE or Tough Matchup</h3>
"""

for _, p in avoid_list.iterrows():
    issue = "😴 BYE" if p['has_bye'] else f"🚫 Tough — vs {p['opponent']}"
    html += f"""
  <div class="sc-avoid">
    <div style="display:flex;justify-content:space-between;align-items:center;">
      <div><strong>{p['name']}</strong>
        <span style="color:#6b7280;font-size:13px;margin-left:6px;">({p['team_abbrev']})</span>
      </div>
      <span class="badge badge-red">{p['avg_2025']:.0f} SC avg</span>
    </div>
    <div style="font-size:13px;color:#991b1b;margin-top:4px;">{issue}</div>
  </div>"""

if len(noted) > 0:
    html += """
  <h3 style="color:#374151;">📝 Analyst Notes</h3>
"""
    for _, p in noted.iterrows():
        note = p['latest_note'][:200] + "..." if len(p['latest_note']) > 200 else p['latest_note']
        html += f"""
  <div class="note-box">
    <strong>{p['name']}</strong> ({p['team_abbrev']}) — {p['avg_2025']:.0f} SC avg
    <div style="margin-top:4px;font-style:italic;color:#374151;">"{note}"</div>
    <div style="font-size:11px;color:#9ca3af;margin-top:4px;">📅 {p['note_date']}</div>
  </div>"""

html += f"""
  <div style="font-size:12px;color:#6b7280;margin-top:12px;">
    😴 BYE Round {next_round}: {', '.join(bye_teams_next) if bye_teams_next else 'None'}
  </div>
</div>

<!-- INJURY REPORT -->
<div class="card">
  <h2>🏥 Full Injury & Suspension Report</h2>
  <p class="section-intro">All players currently listed as injured or suspended. Sorted by team then severity.</p>
  <p class="section-intro">🔴 Long-term/TBC &nbsp; 🟠 4–8 weeks &nbsp; 🟡 2–4 weeks &nbsp; 🟢 Near return</p>
  <table>
    <thead><tr>
      <th>Player</th><th>Team</th><th>Injury</th><th>Return</th><th>Severity</th>
    </tr></thead><tbody>
"""

for _, row in _inj.iterrows():
    html += f"""
    <tr>
      <td style="font-weight:600;">{row['Player']}</td>
      <td>{row['Team']}</td>
      <td>{row['Injury']}</td>
      <td>{row['Return']}</td>
      <td>{severity_icon(row['Return'])}</td>
    </tr>"""

html += f"""
  </tbody></table>
</div>

<!-- FOOTER -->
<div class="footer">
  <p>📊 Data: fixturedownload.com + The Odds API + ZeroTackle + SuperCoach</p>
  <p>🤖 Model: Logistic Regression | 62.4% accuracy on 2025 season (213 matches)</p>
  <p>💰 Bet responsibly | 18+ only | <a href="https://www.gamblinghelponline.org.au" style="color:#6b7280;">gamblinghelponline.org.au</a></p>
  <p style="margin-top:6px;color:#d1d5db;">NRL Analytics Weekly © 2026</p>
</div>

</div></body></html>"""

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"\n🎉 Newsletter saved to index.html!")
print(f"   Round {int(latest_round)} recap + Round {next_round} preview")
print(f"   Sections: Results · Ladder · Trajectory · Key Injuries · Model · Odds · SuperCoach · Full Injuries")
print(f"   → Push index.html to GitHub and you're live!")

✅ Round 8 completed | Generating Round 9 preview
✅ Ladder built
✅ Team trajectories calculated
✅ Odds loaded — 8 matches | Remaining: 472
✅ Model loaded: LR Final — trained 2021-2024, validated 2025
✅ Predictions generated for 8 matches
✅ Injuries: 89 players
✅ SuperCoach tips ready | Key injured players: 26
✅ Fetched 17 team logos from NRL API
Building newsletter HTML...

🎉 Newsletter saved to index.html!
   Round 8 recap + Round 9 preview
   Sections: Results · Ladder · Trajectory · Key Injuries · Model · Odds · SuperCoach · Full Injuries
   → Push index.html to GitHub and you're live!
